<p><font size="6" color='grey'> <b>

Generative KI. Verstehen. Anwenden. Gestalten.
</b></font> </br></p>

<p><font size="5" color='grey'> <b>
M18 | DSPy: RAG-Queries optimieren (Optional)
</b></font> </br></p>

---


In [ ]:
#@title 🛠️ Umgebung einrichten { display-mode: "form" }
!uv pip install --system -q git+https://github.com/ralf-42/GenAI.git#subdirectory=04_modul

# LangSmith Env-Vars VOR allen LangChain-Imports setzen
import os
os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_PROJECT"] = "M18-DSPy-RAG-Optimierung"
os.environ["LANGSMITH_ENDPOINT"] = "https://eu.api.smith.langchain.com"

# ── Projekt-Utilities ─────────────────────────────────────────────────────────
from genai_lib.utilities import (
    check_environment,
    get_ipinfo,
    setup_api_keys,
    mprint,
    mermaid,
    get_model_profile,
    extract_thinking,
    load_prompt,
)

# ── Projekt-Utilities ─────────────────────────────────────────────────────────
from genai_lib.model_config import BASELINE


setup_api_keys(["OPENAI_API_KEY", "LANGSMITH_API_KEY"], create_globals=False)
print()

check_environment()
print()

get_ipinfo()

# 1 | Warum DSPy?
---


LangChain und LangGraph beantworten vor allem die Architekturfrage: Wie werden Modell, Retriever, Tools, Zustand und Tracing verbunden? DSPy beantwortet eine andere Frage: Wie wird ein einzelner LM-Baustein besser, ohne den Prompt immer wieder von Hand umzuschreiben?

Der rote Faden in diesem Notebook ist deshalb bewusst eng: Eine LangChain-nahe RAG-Pipeline hat einen schwachen Query-Rewriter. DSPy optimiert genau diesen Baustein über Trainingsbeispiele und eine Metrik. Der Rest der Anwendung bleibt außerhalb von DSPy.


In [ ]:
#@markdown   <p><font size="4" color='green'>  LangChain + DSPy im RAG-Ablauf</font> </br></p>

diagram = '''
%%{init: {'theme':'forest'}}%%
flowchart LR
    Q["Nutzerfrage"] --> DS1["DSPy Signature<br/>question -> search_query"]
    DS1 --> DS2["DSPy Optimizer<br/>compile mit Beispielen"]
    DS2 --> SQ["optimierte Suchquery"]

    subgraph LC["LangChain RAG aus M06"]
        direction TB
        D["Document"] --> S["RecursiveCharacterTextSplitter"]
        S --> E["OpenAIEmbeddings"]
        E --> C[("Chroma Vectorstore")]
        C --> R["similarity_search_with_score"]
    end

    SQ --> R
    R --> M["Retrieval-Metrik<br/>Top-1 topic == expected_topic"]

    style DS2 fill:#10a37f,stroke:#333,color:#000
    style SQ fill:#10a37f,stroke:#333,color:#000
    style C fill:#FFD700,stroke:#333,color:#000
    style M fill:#ADD8E6,stroke:#333,color:#000
'''

mermaid(diagram, width=1100)


| Dimension | LangChain / LangGraph | DSPy |
|---|---|---|
| Hauptzweck | Orchestrierung: Chains, Agents, RAG, Tools, State | Optimierung einzelner LM-Aufgaben |
| Prompt | Wird explizit geschrieben und versioniert | Entsteht aus `Signature`, Beispielen und Optimizer |
| Evaluation | LangSmith-Datasets, Experimente, Traces | Metrik ist direkt Ziel des Compile-Schritts |
| Stärkster Einsatz | Anwendungspipeline und Integration | Klassifikation, Extraktion, Query-Rewriting, Judge-Bausteine |


<p><font color='darkblue' size="4">
🔗 <b>Viz</b>
</font></p>

- [DSPy](https://editor.p5js.org/ralf.bendig.rb/full/Q_dUC2-lV) visualisiert das Zusammenspiel aus Signature, Optimizer und kompiliertem Modul aus diesem Kapitel interaktiv.

# 2 | Minimaler Wortschatz
---


<p><font color='darkblue' size="4">
✏️ <b>Notiz</b>
</font></p>

Für den Synergie-Case reichen vier Begriffe:

| Begriff | Bedeutung im Beispiel |
|---|---|
| `Signature` | beschreibt Input und Output des Query-Rewriters |
| `Predict` | führt die Signature ohne zusätzliche Strategie aus |
| `metric` | bewertet, ob die erzeugte Suchfrage die Zielbegriffe trifft |
| `Optimizer` | kompiliert ein besseres Modul aus Beispielen und Metrik |


In [ ]:
#@markdown 📦 DSPy installieren { display-mode: "form" }
!uv pip install -q dspy

In [ ]:
import dspy

# BASELINE hat das Format "openai:gpt-5.4-nano" (init_chat_model-Konvention).
# DSPy/LiteLLM erwartet "openai/gpt-5.4-nano" (Slash statt Doppelpunkt).
dspy_model = BASELINE.replace("openai:", "openai/")

# gpt-5.x-Modelle akzeptieren kein temperature=0.0 (DSPy-Standard) - daher temperature=1.0.
dspy.configure(lm=dspy.LM(dspy_model, temperature=1.0))

# 3 | DSPy im Standard-RAG-Workflow?
---



DSPy kann in einen Standard-RAG-Workflow eingebaut werden, sollte dort aber nicht automatisch zur Grundausstattung werden. Für kleine Prototypen, wenige Dokumente oder fehlende Metriken ist DSPy meistens zusätzlicher Overhead. Der Nutzen entsteht erst, wenn ein wiederkehrender Qualitätsengpass messbar ist.

<p><font color='darkblue' size="4">
✨ <b>Empfehlung:</b>
</font></p>

RAG zuerst mit LangChain oder LangGraph sauber aufbauen. Danach Retrieval-Fehler mit einem Evalset oder LangSmith messen. DSPy kommt erst dann dazu, wenn ein einzelner Baustein systematisch verbessert werden soll, zum Beispiel `question -> search_query`, Routing, Klassifikation oder Extraktion.

| Situation | DSPy-Einsatz | Begründung |
|---|---|---|
| Kleiner RAG-Prototyp ohne Evalset | Nicht nötig | Manuelles Prompting ist schneller und verständlicher |
| RAG trifft gelegentlich falsch, aber Fehler sind unklar | Noch warten | Zuerst Fehlerklassen und Metrik definieren |
| Wiederkehrende Retrieval-Fehler bei indirekten Nutzerfragen | Sinnvoll | Query-Rewriter kann mit Beispielen und Metrik optimiert werden |
| Migration auf kleineres oder lokales Modell | Sinnvoll | DSPy kann Prompts und Beispiele gegen Qualitätsmetriken anpassen |
| Produktiver RAG-Service mit Regressionstests | Strategisch | Optimierung wird nachvollziehbar, versionierbar und wiederholbar |

<p><font color='darkblue' size="4">
⚠️ <b>Warnung</b>
</font></p>

DSPy optimiert nicht die ganze Anwendung. Es lohnt sich vor allem für klar abgegrenzte LM-Bausteine mit Eingabe, Ausgabe und Metrik. Ohne Beispiele und Bewertung ist DSPy nur ein weiterer Wrapper und kein systematischer Qualitätshebel.


<p><font color='darkblue' size="4">
🔎 <b>Compiled vs. not compiled</b>
</font></p>

`DSPy ohne Compile` nutzt nur die `Signature` und das Sprachmodell. DSPy weiß also, welche Eingabe und Ausgabe gemeint sind, hat aber noch keine optimierte Prompt- oder Beispielstrategie für diese konkrete Aufgabe.

`DSPy compiled` wurde vorher mit Trainingsbeispielen und einer Optimierungsstrategie erzeugt. Im Mini-Beispiel übernimmt `LabeledFewShot` passende Beispiele aus dem `trainset` und baut daraus eine stärkere Anleitung für `question -> search_query`.

Wichtig: `compiled` bedeutet nicht, dass das Modell neu trainiert wurde. Die Modellgewichte bleiben gleich. DSPy optimiert den Prompt-Kontext um das Modell herum. Deshalb kann `compiled` im Durchschnitt besser sein, aber bei einzelnen Fragen trotzdem schlechter abschneiden, besonders wenn Evalset, Korpus oder Metrik sehr klein sind.

# 4 | Beispiel: Query-Rewriter optimieren
---


Eine einfache RAG-Pipeline scheitert oft nicht am Generator, sondern am Retrieval-Input. Die Nutzerfrage enthält Alltagssprache, der Kurskorpus enthält technische Begriffe. Genau hier ist DSPy sinnvoll: Ein kleiner Query-Rewriter wird mit Beispielen kompiliert und anschließend gegen echten LangChain-Retrieval-Erfolg bewertet.



<p><font color='darkblue' size="4">
🎯 <b>Anwendungsfall</b>
</font></p>

LangChain übernimmt wie in `M06_RAG_LangChain.ipynb` den RAG-Teil: `Document`, `RecursiveCharacterTextSplitter`, `OpenAIEmbeddings`, `Chroma` und `similarity_search_with_score`. DSPy optimiert nur den Übergang von `question -> search_query`. Der Synergie-Case lautet deshalb: LangChain baut die Retrieval-Infrastruktur, DSPy verbessert die Suchfrage, an der Retrieval-Qualität messbar hängt.

Die Testfragen sind bewusst indirekter formuliert und der Mini-Korpus enthält zusätzlich Ablenk-Dokumente. Dadurch wird sichtbar, wann direkte Suche genügt und wann ein kompilierter Query-Rewriter den besseren Top-1-Treffer liefert.


In [ ]:
# ─── Importe  ─────────────────────────────────────────────────────────────
from pathlib import Path
import shutil

import dspy
import pandas as pd
from IPython.display import Markdown, display
from langchain_chroma import Chroma
from langchain_core.documents import Document
from langchain_openai import OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [ ]:
# ─── Mini-Korpus im Stil von M06_RAG_LangChain ───────────────────────────────
raw_documents = [
    Document(
        page_content=(
            "Chunking teilt lange Dokumente in kleinere, überlappende Textabschnitte. "
            "In RAG-Systemen verhindert Chunking, dass zu viel irrelevanter Kontext an das LLM geht."
        ),
        metadata={"source": "m06_rag_chunking", "topic": "chunking"},
    ),
    Document(
        page_content=(
            "Embeddings wandeln Texte in numerische Vektoren um. "
            "Vektordatenbanken nutzen diese Repräsentationen für semantische Ähnlichkeitssuche."
        ),
        metadata={"source": "m06_rag_embeddings", "topic": "embeddings"},
    ),
    Document(
        page_content=(
            "Chroma ist eine Vektordatenbank für Retrieval-Augmented Generation. "
            "LangChain kann Chroma als Vectorstore verwenden und Dokumente mit Scores abrufen."
        ),
        metadata={"source": "m06_rag_chroma", "topic": "vectorstore"},
    ),
    Document(
        page_content=(
            "LCEL verbindet Retriever, Prompt, LLM und StrOutputParser zu einer RAG-Chain. "
            "Der Retriever liefert Kontext, der Prompt formatiert Frage und Dokumente."
        ),
        metadata={"source": "m06_rag_lcel", "topic": "lcel"},
    ),
    Document(
        page_content=(
            "Eine normale Stichwortsuche vergleicht Wörter direkt. "
            "Sie ist einfach, aber bei Synonymen und indirekten Fragen oft unzuverlässig."
        ),
        metadata={"source": "m06_rag_keyword_search", "topic": "keyword_search"},
    ),
    Document(
        page_content=(
            "Ein Prompt ist eine Vorlage für die Modellanfrage. "
            "Gute Prompts strukturieren Rollen, Aufgabe, Kontext und gewünschtes Ausgabeformat."
        ),
        metadata={"source": "m06_rag_prompting", "topic": "prompting"},
    ),
]

text_splitter = RecursiveCharacterTextSplitter(chunk_size=350, chunk_overlap=40)
docs = text_splitter.split_documents(raw_documents)

chroma_dir = Path("chroma_dspy_demo")
shutil.rmtree(chroma_dir, ignore_errors=True)

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
vectorstore = Chroma.from_documents(
    documents=docs,
    embedding=embeddings,
    persist_directory=str(chroma_dir),
)

In [ ]:
# ─── DSPy-Rewriter ───────────────────────────────────────────────────────────
class RewriteSearchQuery(dspy.Signature):
    """Rewrite a learner question into a short technical retrieval query for German RAG course notes."""
    question: str = dspy.InputField()
    search_query: str = dspy.OutputField(
        desc="short technical query using course terms such as Chunking, Embeddings, Chroma, LCEL, Retriever"
    )

trainset = [
    dspy.Example(
        question="Wie verhindere ich, dass zu viel Text im Kontext landet?",
        search_query="Chunking Dokumente Textabschnitte RAG Kontext",
    ).with_inputs("question"),
    dspy.Example(
        question="Wie findet RAG passende Abschnitte nach Bedeutung?",
        search_query="Embeddings Vektoren semantische Ähnlichkeitssuche",
    ).with_inputs("question"),
    dspy.Example(
        question="Wo speichert LangChain die Vektoren für die Suche?",
        search_query="Chroma Vectorstore Vektordatenbank similarity_search_with_score",
    ).with_inputs("question"),
    dspy.Example(
        question="Wie werden Retriever, Prompt und Modell verbunden?",
        search_query="LCEL Retriever Prompt LLM StrOutputParser RAG Chain",
    ).with_inputs("question"),
]

# Die Testfragen sind absichtlich indirekt formuliert.
# Der erwartete Treffer hängt an Kursbegriffen, die nicht immer in der Frage stehen.
devset = [
    {
        "question": "Mein PDF ist zu lang und die Antwort nimmt zu viel Nebentext mit. Was ist der passende Vorverarbeitungsschritt?",
        "expected_topic": "chunking",
    },
    {
        "question": "Wie erkennt die Suche verwandte Inhalte, obwohl die Formulierungen unterschiedlich sind?",
        "expected_topic": "embeddings",
    },
    {
        "question": "Wo liegt der Index, den der Retriever später für die Fundstellen abfragt?",
        "expected_topic": "vectorstore",
    },
    {
        "question": "Wie fließen gefundene Ausschnitte, Vorlage und Sprachmodell zu einer Antwort zusammen?",
        "expected_topic": "lcel",
    },
]

baseline_rewriter = dspy.Predict(RewriteSearchQuery)
compiled_rewriter = dspy.LabeledFewShot(k=len(trainset)).compile(
    dspy.Predict(RewriteSearchQuery),
    trainset=trainset,
)

In [ ]:
# ─── Retrieval-Auswertung ────────────────────────────────────────────────────

def retrieve_top_topic(search_query: str):
    docs_with_scores = vectorstore.similarity_search_with_score(search_query, k=1)
    top_doc, score = docs_with_scores[0]
    return top_doc.metadata["topic"], top_doc.metadata["source"], score, top_doc.page_content


def direct_query(question: str) -> str:
    return question


def baseline_dspy_query(question: str) -> str:
    return baseline_rewriter(question=question).search_query


def compiled_dspy_query(question: str) -> str:
    return compiled_rewriter(question=question).search_query


def collect_retrieval_results(examples) -> pd.DataFrame:
    variants = [
        ("Direkt", direct_query),
        ("DSPy ohne Compile", baseline_dspy_query),
        ("DSPy compiled", compiled_dspy_query),
    ]
    rows = []

    for question_id, example in enumerate(examples, 1):
        for variant, query_builder in variants:
            search_query = query_builder(example["question"])
            topic, source, score, preview = retrieve_top_topic(search_query)
            is_match = topic == example["expected_topic"]

            rows.append({
                "Frage": question_id,
                "Originalfrage": example["question"],
                "Erwartet": example["expected_topic"],
                "Variante": variant,
                "Suchquery": search_query,
                "Top-1-Treffer": topic,
                "Quelle": source,
                "Score": score,
                "Bewertung": "OK" if is_match else "Fehltreffer",
                "Auszug": preview[:120] + "...",
                "Treffer": int(is_match),
            })

    return pd.DataFrame(rows)


def show_results(results_df: pd.DataFrame) -> pd.DataFrame:
    display_columns = [
        "Frage",
        "Originalfrage",
        "Erwartet",
        "Variante",
        "Suchquery",
        "Top-1-Treffer",
        "Quelle",
        "Score",
        "Bewertung",
    ]
    text_columns = [
        "Originalfrage",
        "Erwartet",
        "Variante",
        "Suchquery",
        "Top-1-Treffer",
        "Quelle",
        "Bewertung",
    ]

    display(Markdown("### Gegenüberstellung pro Suchquery"))
    display(
        results_df[display_columns]
        .style
        .hide(axis="index")
        .format({"Score": "{:.4f}"})
        .set_properties(subset=text_columns, **{"text-align": "left"})
        .set_properties(subset=["Frage", "Score"], **{"text-align": "right"})
        .set_table_styles([
            {"selector": "th", "props": [("text-align", "left")]},
            {"selector": "td", "props": [("vertical-align", "top")]},
        ])
    )

    summary_df = (
        results_df
        .groupby("Variante", as_index=False)
        .agg(Treffer=("Treffer", "sum"), Fragen=("Frage", "nunique"))
    )
    summary_df["Retrieval-Score"] = summary_df["Treffer"].astype(str) + "/" + summary_df["Fragen"].astype(str)
    summary_df = summary_df[["Variante", "Retrieval-Score", "Treffer", "Fragen"]]

    display(Markdown("### Zusammenfassung"))
    display(
        summary_df
        .style
        .hide(axis="index")
        .set_properties(subset=["Variante", "Retrieval-Score"], **{"text-align": "left"})
        .set_properties(subset=["Treffer", "Fragen"], **{"text-align": "right"})
        .set_table_styles([
            {"selector": "th", "props": [("text-align", "left")]},
            {"selector": "td", "props": [("vertical-align", "top")]},
        ])
    )

    return summary_df


In [ ]:
# ─── Main ─────────────────────────────────────────────────────────────────────────
results_df = collect_retrieval_results(devset)
summary_df = show_results(results_df)

score_by_variant = dict(zip(summary_df["Variante"], summary_df["Treffer"]))
assert score_by_variant["DSPy compiled"] >= score_by_variant["Direkt"], (
    "Der kompilierte Rewriter sollte das direkte Retrieval nicht verschlechtern."
)



<p><font color='darkblue' size="4">
⚠️ <b>Hinweis</b>
</font></p>

Das Beispiel erzeugt OpenAI-Embeddings und ruft für den DSPy-Rewriter ein Sprachmodell auf. Die Scores können sich bei Modellwechseln leicht ändern. In produktionsnahen Setups würde man die Beispiele als LangSmith- oder DSPy-Evalset versionieren und deutlich mehr Testfragen verwenden.


# 5 | Prompt-Optimierung als Messproblem
---


Automatic Prompt Optimization behandelt Prompt Engineering nicht als Bauchgefühl, sondern als einfachen Optimierungszyklus: Ein Prompt wird an Beispielen getestet, Fehler werden gesammelt, daraus entstehen neue Prompt-Kandidaten, und nur messbar bessere Varianten bleiben übrig.

Gute Prompts entstehen nicht durch einmaliges Formulieren, sondern durch kleine Experimente mit klarer Metrik.

In [ ]:
#@markdown   <p><font size="4" color='green'>  Prompt-Optimierung als Eval-Schleife</font> </br></p>

diagram = '''
%%{init: {'theme':'forest'}}%%
flowchart LR
    P["Basis-Prompt"] --> E["Evalset<br/>Beispiele + erwartete Wirkung"]
    E --> R["Ausführen<br/>Antworten oder Retrieval-Treffer"]
    R --> M["Metrik<br/>Score, Treffer, Fehlerklasse"]
    M --> K["Kritik<br/>textueller Gradient"]
    K --> C["Prompt-Kandidaten<br/>APE/APO/GRIPS"]
    C --> A{"Besser?"}
    A -- "ja" --> B["neuer bester Prompt"]
    A -- "nein" --> P
    B --> R

    style M fill:#ADD8E6,stroke:#333,color:#000
    style K fill:#FFD700,stroke:#333,color:#000
    style B fill:#10a37f,stroke:#333,color:#000
'''

mermaid(diagram, width=1100)

| Abkürzung    | Ausgeschrieben                                                            | Bedeutung                                                                                                                                                                          |
| ------------ | ------------------------------------------------------------------------- | ---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------- |
| **APE**      | **Automatic Prompt Engineer**                                             | Ein Verfahren, das automatisch mehrere Prompt-Varianten erzeugt, diese bewertet und die beste auswählt. <br>Das folgt dem Muster: *Generate → Score → Select*.                         |
| **APO**      | **Automatic Prompt Optimization** (oft auch *Automatic Prompt Optimizer*) | Optimiert Prompts iterativ. Statt numerischer Gradienten werden Fehler in natürlicher Sprache beschrieben („textual gradients“) <br>und zur Verbesserung des nächsten Prompts genutzt. |
| **GRIPS**    | **Gradient-free, Edit-based Prompt Search**                               | Ein Verfahren, das Prompts durch kleine Editieroperationen verändert (z. B. Löschen, Austauschen, Umformulieren oder Hinzufügen von Text), <br>ohne Gradientenverfahren zu verwenden.  |
| **Hold-out** | keine Abkürzung                                                           | Bezeichnet einen zurückgehaltenen Testdatensatz, der während der Optimierung nicht verwendet wird und erst zur unabhängigen Bewertung dient.                                       |


<p><font color='darkblue' size="4">
📌 <b>Merksatz</b>
</font></p>

DSPy automatisiert einen Teil dieser Schleife. Trotzdem bleibt die wichtigste Designentscheidung menschlich: Welche Beispiele zählen, welche Metrik misst die gewünschte Qualität, und wann ist eine Verbesserung nur auf dem Trainingsset besser?

Im folgenden Mini-Beispiel ist deshalb nicht der generierte Prompt das eigentliche Lernziel, sondern die messbare Kopplung aus `question -> search_query`, Retrieval-Erfolg und Testfragen.

# A | Aufgabe
---


**Grundlagen** — Eine weitere Testfrage in `devset` ergänzen und ein passendes `expected_topic` vergeben. Danach prüfen, ob direkte Suche und DSPy-Rewriter denselben Treffer liefern.



**Aufbau** — Den Mini-Korpus um ein fünftes Dokument erweitern, z. B. zu `LangSmith` oder `Reranking`, und dafür ein Trainingsbeispiel ergänzen.



**Vertiefung** — Den optimierten DSPy-Rewriter vor die RAG-Chain aus `M06_RAG_LangChain.ipynb` setzen und die Ergebnisse mit LangSmith vergleichen.


# B | Dokumente zum Weiterlesen
---



📚 Ergänzende Artikel aus der Kurs-Dokumentation:

- [Prompt Engineering](https://ralf-42.github.io/GenAI/05-prompting-rag/prompt-engineering.html)
- [Context Engineering](https://ralf-42.github.io/GenAI/05-prompting-rag/context-engineering.html)
- [RAG-Konzepte](https://ralf-42.github.io/GenAI/05-prompting-rag/rag-konzepte.html)
